# Reproduction of manuscript Figure 5

Seven-site FMO benchmark at 77 K and 300 K. The Hamiltonian is supplied from `manuscript_models.py`; the same generic N-site solver is used as for the three-site calculations.

In [ ]:
from pathlib import Path
from dataclasses import replace
import numpy as np
import matplotlib.pyplot as plt
from QME import Config, run_system
from manuscript_models import fmo_7site_hamiltonian

OUT = Path("results")
OUT.mkdir(exist_ok=True)

H = fmo_7site_hamiltonian()
lambda_i = np.full(H.shape[0], 35.0)
cfg77 = Config(temperature_k=77.0, tau_g_ps=0.05, nmax=10_000, dt_ps=1.0e-4, seed=12345)
cfg300 = replace(cfg77, temperature_k=300.0)


In [ ]:
R = {}
for cfg in (cfg77, cfg300):
    T = int(cfg.temperature_k)
    for rep in ("present", "cmrt", "forster"):
        R[rep, T] = run_system(
            H, lambda_i, cfg, rep, initial_site=0,
            output_dir=OUT, tag=f"Fig5_{rep}_{T}K",
        )

    print(f"{T} K")
    for rep in ("present", "cmrt", "forster"):
        print(f"  {rep:7s} Vave = {R[rep,T]['basis']['vave']:.6f} cm^-1")

In [ ]:
PAPER_COLORS_FMO = ["#ee2c2a", "#2555a6", "#6dbe44", "#947fbb", "#7ac4dd", "#ece94e", "#b7b7ba"]

panels = [
    ("present", 77, "(a) Present at 77 K"),
    ("cmrt", 77, "(b) CMRT at 77 K"),
    ("forster", 77, "(c) Förster at 77 K"),
    ("present", 300, "(d) Present at 300 K"),
    ("cmrt", 300, "(e) CMRT at 300 K"),
    ("forster", 300, "(f) Förster at 300 K"),
]

fig, axs = plt.subplots(2, 3, figsize=(12, 7), sharex=True, sharey=True)
for ax, (rep, T, title) in zip(axs.flat, panels):
    r = R[rep, T]
    for i in range(H.shape[0]):
        ax.plot(
            r["time_ps"], r["populations"][:, i],
            color=PAPER_COLORS_FMO[i], linewidth=1.1, label=str(i + 1),
        )
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="Time (ps)", ylabel="Probability", title=title)
    ax.set_box_aspect(1)
    ax.legend(frameon=False, ncol=2)

fig.tight_layout()
fig.savefig(OUT / "Fig5.pdf", bbox_inches="tight")
plt.show()
